# 99장 · 시간 초과의 재시도와 중단 조건

시간 초과 뒤 다시 시도할 수 있어도 같은 요청을 끝없이 보낼 수는 없습니다. 실행 기록에서 재시도 횟수와 중단 사유를 읽습니다.

검색 시간 초과 뒤 재시도하는 요청과 근거 부족으로 정상 종료한 요청을 비교합니다. 상태·시도 번호·종료 사유를 기록해 복구 경로를 추적합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 일시 오류 후 재시도 기록

복구 기록 F99-recover　Q01의 첫 시도에서 가상 도구가 TimeoutError를 냈습니다. 작업을 paused에 두었다가 원래 요청 ID로 두 번째 시도를 해서 마쳤습니다. 여기서 시간 초과는 교육 예제에서 의도적으로 일으킨 실패입니다.


### 첫 타임아웃에서 두 번째 실행으로 돌아가는 간선

제공된 이벤트 기록에서 retry는 첫 시도에 속합니다. 다음 attempt가 찍힐 때 시도 번호가 2로 바뀝니다.

1. attempt · attempt = 1
2. paused · error_type = TimeoutError
3. retry · attempt = 1
4. attempt · attempt = 2
5. completed · event = completed


### 셀 01 · 재시도 이벤트 순서

제공 실행 기록에서 F99-recover의 사건만 꺼낸 자료입니다. attempts를 무조건 1로 다시 시작하면 원인을 놓칩니다.

F99-recover는 한 요청을 두 번 시도한 예입니다. 첫 시도는 TimeoutError 뒤에 멈추고 두 번째 시도에서 마칩니다. 팀 기록에서는 두 줄에 같은 요청 ID와 서로 다른 시도 번호를 적으세요. 이 셀의 events만으로 ID가 일치했는지까지 확인할 수는 없습니다.


In [1]:
events = [{'event': 'attempt', 'attempt': 1},
 {'event': 'paused', 'attempt': 1, 'error_type': 'TimeoutError'},
 {'event': 'retry', 'attempt': 1},
 {'event': 'attempt', 'attempt': 2},
 {'event': 'completed', 'attempt': 2, 'error_type': ''}]
for event in events:
    print(event["attempt"], event["event"])

1 attempt
1 paused
1 retry
2 attempt
2 completed


시도 1에는 attempt·paused·retry가, 시도 2에는 attempt·completed가 보입니다. retry 행의 1은 첫 실패를 마치고 재시도를 결정한 위치입니다. 실제 두 번째 호출은 다음 attempt에서 시작합니다.

attempt 번호가 1에서 2로 늘어난 시점과 paused 상태에서 다시 시작한 시점을 함께 읽습니다. 중간에 멈춘 요청을 새 작업으로 기록하면 팀은 재시도 횟수를 과소집계할 수 있습니다. 마지막 이벤트에는 성공 여부와 중단 사유를 구분해 남깁니다.


#### 재시도 기록에서 끝을 찾는 법

```mermaid
flowchart TD
a["attempt"] --> q{"처리 결과?"}
q -->|성공| c["completed"]
q -->|시간 초과| r{"시도 횟수 남음?"}
r -->|예| b["retry"] --> a
r -->|아니오| e["attempt_limit"]
```

기록의 attempt 번호는 첫 시도부터 셉니다. 일시적 실패 후 다시 시도한 사례와 마지막 시도까지 실패한 사례의 종료 사유를 비교하세요.

## 반복 실패의 종료 사유

실행 상한 예시　최대 시도 2회 / 둘째 시도에서도 시간 초과 / 최종 사건 attempt_limit / 사용자 안내 시안 “지금은 검색을 마치지 못했습니다. 잠시 뒤 다시 요청해 주세요.” 실패를 no_evidence로 보고하면 사용자는 원문 자료가 없다고 잘못 이해합니다. 결제나 예약이 있는 팀은 재시도가 같은 동작을 두 번 수행하지 않는지도 별도로 설계해야 합니다.


### 과제 02 · 선택 연습 · 종료 사유 기록

F99-limit의 마지막 사건을 읽어 화면 담당자에게 횟수와 원인을 전하세요.


In [2]:
final_event = {'event': 'attempt_limit', 'attempt': 2, 'error_type': 'TimeoutError'}
# 마지막 사건·횟수·오류명을 표시하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>final_event = {&#x27;event&#x27;: &#x27;attempt_limit&#x27;, &#x27;attempt&#x27;: 2, &#x27;error_type&#x27;: &#x27;TimeoutError&#x27;}
print(final_event[&quot;event&quot;], final_event[&quot;attempt&quot;], final_event[&quot;error_type&quot;])</code></pre>
<p>실행 결과</p><pre><code>attempt_limit 2 TimeoutError</code></pre>
<p>attempt_limit, 2, TimeoutError입니다. 추가 호출을 열어 두지 않고 마지막 실패를 다시 볼 수 있습니다.</p>
</details>


#### 재시도 상한을 읽는 방법

<mark>첫 시도도 시도 횟수에 포함됩니다.</mark> 이벤트에서 attempt 번호와 retry 기록을 따로 세어 보세요. 두 번째에도 시간 초과가 났다면 마지막 오류와 중단 사유를 남겨, 세 번째 실행을 기다리지 않게 합니다.

선수 개념 · 27장 LangGraph 조건부 간선과 재검색

## 시간 초과를 한 번 겪은 검색을 같은 요청으로 다시 실행하기

앞 셀의 events는 이미 남은 기록입니다. 이번에는 호출하는 동안 검색 대역이 실제로 TimeoutError를 한 번 일으키게 하고, 같은 요청 ID로 두 번째 시도가 이어지는지 봅니다.

### 셀 04 · 호출 횟수에 상한을 둔 로컬 실행

처음 시도는 실패하고 다음 시도는 RULE-A를 돌려주는 수업용 함수를 만듭니다. 작업이 조회라서 이 예시는 동일 요청으로 재시도할 수 있습니다.

In [3]:
def read_after_delay(attempt):
    if attempt == 1:
        raise TimeoutError("첫 조회 지연")
    return "RULE-A"

def run_lookup(request_id, reader, limit=2):
    for attempt in range(1, limit + 1):
        try:
            return request_id, attempt, reader(attempt)
        except TimeoutError:
            if attempt == limit:
                return request_id, attempt, "attempt_limit"

print(*run_lookup("F99-recover", read_after_delay))
print(*run_lookup("F99-stop", read_after_delay, limit=1))

F99-recover 2 RULE-A
F99-stop 1 attempt_limit


첫 요청은 두 번째 호출에서 RULE-A를 얻습니다. 두 번째 요청은 한 번만 시도하도록 설정해 attempt_limit에서 끝났습니다. 실제 네트워크 시간을 기다리지는 않습니다.

### 팀 도구에 맞춰 재시도를 허용할지 정하기

조회가 아니라 상태를 바꾸는 도구라면 응답이 늦어도 첫 작업이 이미 끝났을 수 있습니다. 같은 요청을 다시 받았을 때 이전 결과를 조회할 수 있는지 먼저 확인하고, 그런 보장이 없으면 재실행 전에 담당자가 확인하도록 설계합니다.

<mark>시간 초과는 자료가 없다는 결론을 내릴 근거가 아닙니다.</mark> 첫 호출에 응답이 없었어도 둘째 호출은 같은 자료에서 RULE-A를 읽었습니다.

100장 저장 설계에는 한 요청의 재시도와 서로 다른 사용자의 동명 세션을 구분할 키가 필요합니다.

## 돌아보기

상태를 넘길 때 마지막 사건과 시도 횟수, 다시 시작할 수 있는지를 남기세요. 100장에서는 완료한 질문과 답을 사용자별 세션에 어떻게 보관하는지 살펴봅니다.
